# Crypto Candlestick-CNN — paper trading (Stage 5)

Trades the recommended config **I20/R20** (20-day image, 20-day hold) forward on real market data.
This is the config the report recommends for live use: it rebalances every 20 days, so it keeps most
of its edge even after heavy trading costs (Sharpe +0.71 at 50 bps/side, vs +0.25 for the 5-day
version).

**Read this before trusting any number it prints.**

- **This is PAPER trading, not a live-trading system.** Fills are simulated at the daily close.
  There is no order-book slippage model beyond the cost grid, **no borrow-availability check on the
  short leg**, and no exchange/counterparty risk. It measures the strategy; it does not trade it.
- **The study never saved model weights** (only test predictions), so this notebook trains and saves
  a production 5-member ensemble **once** on data up to a cutoff you choose, then reuses it.
- **No look-ahead:** the model is trained only on data *before* the simulation window, and every
  score at date *t* uses only bars up to and including *t*.
- **The short leg is the honest weak point.** Shorting small alts is expensive or often impossible.
  `SHORT_LEG = False` runs a long-only variant, and the long-leg / short-leg P&L are always reported
  separately so you can see how much of the edge survives without shorting.
- **To trade genuinely forward in real time:** re-run `crypto_data_prep.ipynb` first (it fetches the
  latest klines from Binance), which refreshes `crypto_market_data_2018_2026.parquet`; then re-run
  this notebook. `todays_book()` appends the current target portfolio to a log, so repeating this over
  weeks builds a real out-of-sample paper record.

**Writes only** `crypto_models/` (saved weights) and `crypto_papertrade_log.csv`. It never touches the
study's artifacts or the equities pipeline.

## 0. Config — trade I20/R20

In [ ]:
import os, time, json, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings; warnings.filterwarnings('ignore')
try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception:
    HAVE_PLT = False

# ---- the config being traded (recommended in the report) ----
N_DAYS   = 20          # image size
HORIZON  = 20          # holding period (calendar days)
ANNUALIZER = 365.0     # crypto is 24/7

# ---- production training ----
TRAIN_CUTOFF = '2023-12-31'   # train ONLY on data up to here; simulate strictly after it.
                              # For genuine live use set this to (today - a few weeks) after
                              # refreshing the parquet, so the model never sees the traded period.
ENSEMBLE_SIZE = 5
BATCH_SIZE = 128
EPOCHS     = 100
LR         = 1e-5
PATIENCE   = 2
TRAIN_FRAC = 0.7
SHARED_SPLIT_SEED = 42

# ---- trading mechanics ----
SIM_START  = '2024-01-01'     # must be > TRAIN_CUTOFF (asserted below)
SIM_END    = '2026-06-30'
COST_BPS   = 25               # per side, per rebalance; 25 is a realistic mid for top-100 crypto
MIN_NAMES  = 20               # need >=20 coins to form deciles
SHORT_LEG  = True             # False -> long-only (hold the top decile); honest if you can't short
CAPITAL    = 100_000.0

# ---- paths (all crypto-namespaced; nothing here is shared with the study or equities) ----
PARQUET_PATH = './crypto_market_data_2018_2026.parquet'
IMG_DIR      = './crypto_image_store'      # existing 20-day store is reused for FAST training
MODEL_DIR    = './crypto_models'
LOG_CSV      = './crypto_papertrade_log.csv'
os.makedirs(MODEL_DIR, exist_ok=True)

# ---- device ----
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

MA_LAGS   = [3, 5, 10, 20, 50, 100, 200]
MA_COLS   = [f'MA{L}' for L in MA_LAGS]
META_COLS = ['MOM','STR','WSTR'] + MA_COLS
DROP_INCOMPLETE = True

assert pd.Timestamp(SIM_START) > pd.Timestamp(TRAIN_CUTOFF), \
    "SIM_START must be strictly after TRAIN_CUTOFF (no look-ahead)."

df_market = pd.read_parquet(PARQUET_PATH)
df_market['Date'] = pd.to_datetime(df_market['Date'])
print(f"device={DEVICE} | trading I{N_DAYS}/R{HORIZON} | train<= {TRAIN_CUTOFF} | "
      f"sim {SIM_START}..{SIM_END} | cost {COST_BPS}bps/side | short_leg={SHORT_LEG}")
print(f"parquet: {len(df_market):,} rows, {df_market.Symbol.nunique()} assets, "
      f"through {df_market.Date.max().date()}")

## 1. Image + model (verbatim from the study)

In [ ]:
from PIL import Image, ImageDraw

def _height(n): return 32 if n == 5 else (64 if n == 20 else 96)
def img_shape(n): return _height(n), n * 3

def generate_image_uint8(window_df, n_days):
    height, width = img_shape(n_days)
    img = Image.new('L', (width, height), color=0)
    if len(window_df) == 0: return np.zeros((height, width), dtype=np.uint8)
    first_close = window_df['Close'].iloc[0]
    if first_close == 0 or np.isnan(first_close): return np.zeros((height, width), dtype=np.uint8)
    draw = ImageDraw.Draw(img)
    prices  = window_df[['Open','High','Low','Close']].values / first_close
    ma      = window_df['Close'].rolling(window=n_days, min_periods=1).mean().values / first_close
    volumes = window_df['Volume'].values
    max_p, min_p = np.nanmax(prices), np.nanmin(prices); max_v = np.nanmax(volumes)
    if max_p == min_p: max_p = min_p + 1e-5
    if max_v == 0:     max_v = 1e-5
    price_h = int(height * 0.8); vol_h = height - price_h
    def p2y(p): return int(np.clip(price_h-1-(p-min_p)/(max_p-min_p)*(price_h-1), 0, price_h-1))
    def v2y(v): return int(np.clip(height-1-(v/max_v)*(vol_h-1), price_h, height-1))
    ma_pts = []
    for i in range(len(window_df)):
        x = i*3 + 1
        oy, hy, ly, cy = p2y(prices[i,0]), p2y(prices[i,1]), p2y(prices[i,2]), p2y(prices[i,3])
        draw.line([(x,hy), (x,ly)], fill=255, width=1)
        draw.line([(x-1,oy), (x,oy)], fill=255, width=1)
        draw.line([(x,cy), (x+1,cy)], fill=255, width=1)
        draw.line([(x,height-1), (x,v2y(volumes[i]))], fill=255, width=1)
        ma_pts.append((x, p2y(ma[i])))
    if len(ma_pts) > 1: draw.line(ma_pts, fill=255, width=1)
    return np.array(img, dtype=np.uint8)

def xavier_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super().__init__()
        if n_days == 5:    blocks, vs, vd = 2, 1, 1
        elif n_days == 20: blocks, vs, vd = 3, 3, 2
        else:              blocks, vs, vd = 4, 3, 3
        layers, in_c, out_c = [], 1, 64
        for i in range(blocks):
            stride   = (vs, 1) if i == 0 else (1, 1)
            dilation = (vd, 1) if i == 0 else (1, 1)
            layers += [nn.Conv2d(in_c, out_c, (5,3), stride=stride, padding=(2,1), dilation=dilation),
                       nn.BatchNorm2d(out_c), nn.LeakyReLU(0.01), nn.MaxPool2d((2,1))]
            in_c, out_c = out_c, out_c*2
        self.features = nn.Sequential(*layers)
        H, W = img_shape(n_days)
        flat = self.features(torch.zeros(1,1,H,W)).view(1,-1).size(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(flat, 2))
        self.apply(xavier_init)
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))
print("model defined")

## 2. Train & save the production ensemble (once)

Trains 5 `TrendCNN(20)` members on data **≤ `TRAIN_CUTOFF`**, reusing the existing 20-day image store
(`crypto_image_store/raw_n20.dat`) so no images have to be re-rendered. Weights are saved to
`crypto_models/`. If they already exist the training is skipped — so this is a one-time cost of a few
minutes; re-runs are instant.

The store / label machinery is a verbatim copy of `crypto_train.ipynb`, so the images and labels are
identical to the study.

In [ ]:
def store_paths(n): return (os.path.join(IMG_DIR, f'raw_n{n}.dat'),
                            os.path.join(IMG_DIR, f'meta_n{n}.parquet'))

def align_store(m, df):
    pos_df = (df.sort_values(['Symbol','Date'])
                .assign(pos=lambda x: x.groupby('Symbol').cumcount(),
                        end_date=lambda x: x['Date'].dt.strftime('%Y-%m-%d'))
                [['Symbol','end_date','pos']].rename(columns={'Symbol':'symbol'}))
    mm = m.merge(pos_df, on=['symbol','end_date'], how='left')
    return (mm['pos'].notna() & (mm['pos'].values == mm['end_idx'].values)).values

def load_store(n, df):
    dat, meta = store_paths(n)
    if not (os.path.exists(dat) and os.path.exists(meta)):
        raise FileNotFoundError(f"image store for n={n} missing; run crypto_train.ipynb first")
    m = pd.read_parquet(meta).reset_index(drop=True)
    H, W = img_shape(n); N = len(m)
    if os.path.getsize(dat) != N*H*W: raise RuntimeError("store size mismatch")
    m['_row'] = np.arange(N)
    if DROP_INCOMPLETE:
        m = m[m['end_idx'].values >= (n-1)].reset_index(drop=True)
    ok = align_store(m, df); m = m[ok].reset_index(drop=True)
    return {'dat': dat, 'shape': (N,1,H,W), 'meta': m}

def labels_upto(store, df, horizon, cutoff):
    # Rows + binary labels for training, restricted to Date <= cutoff. Exact-H-day span enforced.
    m = store['meta']
    closes, dates = {}, {}
    for s, g in df.groupby('Symbol'):
        g = g.sort_values('Date')
        closes[s] = g['Close'].values.astype(np.float64)
        dates[s]  = g['Date'].values.astype('datetime64[D]')
    fwd = np.full(len(m), np.nan)
    for s, pos in m.groupby('symbol').indices.items():
        c, dt = closes.get(s), dates.get(s)
        if c is None: continue
        ei = m['end_idx'].values[pos]; tgt = ei + horizon
        ok = tgt < len(c); tgt_c = np.minimum(tgt, len(c)-1)
        span = (dt[tgt_c] - dt[ei]).astype('timedelta64[D]').astype(int)
        exact = ok & (span == horizon)
        with np.errstate(invalid='ignore', divide='ignore'):
            fwd[pos] = np.where(exact & (c[ei] != 0), c[tgt_c]/c[ei] - 1.0, np.nan)
    d = m.copy(); d['Fwd_Ret'] = fwd; d['end_date'] = pd.to_datetime(d['end_date'])
    d = d[(~d['Fwd_Ret'].isna()) & (d['end_date'] <= pd.Timestamp(cutoff))].reset_index(drop=True)
    return d['_row'].values, (d['Fwd_Ret'].values > 0).astype(np.int64)

class ImgDataset(Dataset):
    def __init__(self, dat, shape, rows, labels=None):
        self.dat=dat; self.shape=tuple(int(x) for x in shape)
        self.rows=np.asarray(rows); self.labels=labels; self._mm=None
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if self._mm is None:
            self._mm = np.memmap(self.dat, dtype=np.uint8, mode='r', shape=self.shape)
        img = torch.from_numpy(np.ascontiguousarray(self._mm[int(self.rows[i])]))
        return img if self.labels is None else (img, int(self.labels[i]))

def prod_path(seed): return os.path.join(MODEL_DIR, f'prod_I{N_DAYS}_R{HORIZON}_s{seed}.pt')

def train_and_save(rows, labels, seed):
    out = prod_path(seed)
    if os.path.exists(out):
        print(f"  member s{seed}: cached -> skip"); return
    torch.manual_seed(seed); np.random.seed(seed)
    idx = np.arange(len(rows)); np.random.default_rng(SHARED_SPLIT_SEED).shuffle(idx)
    cut = int(TRAIN_FRAC*len(idx)); tr, va = idx[:cut], idx[cut:]
    dev = torch.device(DEVICE)
    l_tr = DataLoader(ImgDataset(store['dat'], store['shape'], rows[tr], labels[tr]),
                      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    l_va = DataLoader(ImgDataset(store['dat'], store['shape'], rows[va], labels[va]),
                      batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = TrendCNN(N_DAYS).to(dev)
    crit = nn.CrossEntropyLoss(); opt = optim.Adam(model.parameters(), lr=LR)
    best, wait, t0 = float('inf'), 0, time.time()
    best_state = None
    for ep in range(EPOCHS):
        model.train(); tl=0.0
        for x,y in l_tr:
            x=x.to(dev).float().div_(255.0); y=y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(x),y); loss.backward(); opt.step(); tl+=loss.item()
        model.eval(); vl=0.0
        with torch.no_grad():
            for x,y in l_va:
                x=x.to(dev).float().div_(255.0); y=y.to(dev)
                vl+=crit(model(x),y).item()
        vl/=max(1,len(l_va))
        if vl<best: best,wait=vl,0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:       wait+=1
        if wait>=PATIENCE: break
    torch.save(best_state, out)
    print(f"  member s{seed}: {ep+1} epochs, best val {best:.4f}, {time.time()-t0:.0f}s -> {os.path.basename(out)}")

store = load_store(N_DAYS, df_market)
rows, labels = labels_upto(store, df_market, HORIZON, TRAIN_CUTOFF)
print(f"training rows (Date<= {TRAIN_CUTOFF}): {len(rows):,} | positive-label rate {labels.mean():.3f}")
print(f"training {ENSEMBLE_SIZE} members on {DEVICE} (one-time; cached afterwards)...")
for i in range(ENSEMBLE_SIZE):
    train_and_save(rows, labels, 2000 + i)
print("production ensemble ready.")

## 3. Load the ensemble and score any date (leak-free, renders images on the fly)

In [ ]:
def load_ensemble():
    dev = torch.device(DEVICE); models=[]
    for i in range(ENSEMBLE_SIZE):
        p = prod_path(2000+i)
        if not os.path.exists(p): raise FileNotFoundError(f"missing {p}; run the training cell")
        m = TrendCNN(N_DAYS).to(dev); m.load_state_dict(torch.load(p, map_location=dev)); m.eval()
        models.append(m)
    return models

# pre-group the panel once for fast window lookups
PANEL_BY_SYM = {s: g.sort_values('Date').reset_index(drop=True)
                for s, g in df_market.groupby('Symbol')}
CLOSE_BY_SYM = {s: g.set_index('Date')['Close'] for s, g in PANEL_BY_SYM.items()}

def _window(sym, asof):
    # The N_DAYS bars ending exactly at asof for this symbol, or None. Uses only data <= asof.
    g = PANEL_BY_SYM.get(sym)
    if g is None: return None
    j = g.index[g['Date'] == pd.Timestamp(asof)]
    if len(j) == 0: return None
    e = int(j[0])
    if e < N_DAYS - 1: return None
    return g.iloc[e-N_DAYS+1 : e+1]

def score_asof(models, asof):
    # CNN long-probability for every universe coin trading at asof. Renders images live.
    asof = pd.Timestamp(asof)
    uni = df_market[(df_market['Date'] == asof) & (df_market['InUniverse'])]['Symbol'].tolist()
    syms, imgs = [], []
    for s in uni:
        w = _window(s, asof)
        if w is None or len(w) < N_DAYS: continue
        syms.append(s); imgs.append(generate_image_uint8(w[['Open','High','Low','Close','Volume']], N_DAYS))
    if not syms: return pd.Series(dtype=float)
    dev = torch.device(DEVICE)
    x = torch.from_numpy(np.stack(imgs)[:, None, :, :]).to(dev).float().div_(255.0)
    with torch.no_grad():
        probs = np.mean([torch.softmax(m(x), 1)[:,1].cpu().numpy() for m in models], axis=0)
    return pd.Series(probs, index=syms).sort_values(ascending=False)

def realized_fwd(sym, asof, horizon=HORIZON):
    # Actual return from close(asof) to close(asof+horizon days); NaN if not both present.
    c = CLOSE_BY_SYM.get(sym)
    if c is None: return np.nan
    t0, t1 = pd.Timestamp(asof), pd.Timestamp(asof) + pd.Timedelta(days=horizon)
    if t0 not in c.index or t1 not in c.index: return np.nan
    p0 = c.loc[t0]; return c.loc[t1]/p0 - 1.0 if p0 else np.nan

models = load_ensemble()
print(f"ensemble loaded ({len(models)} members). Sanity score on the last date:")
_last = df_market['Date'].max()
_p = score_asof(models, _last)
print(f"  {_last.date()}: scored {len(_p)} coins | prob range [{_p.min():.3f}, {_p.max():.3f}]")

## 4. Forward simulation

Walk the 20-day rebalance grid over `[SIM_START, SIM_END]` (strictly after the training cutoff). At
each date: score the universe, long the top decile, short the bottom decile (equal-weight), hold 20
days, settle from realised prices, charge `2 × COST_BPS`. Non-overlapping, so returns compound
cleanly.

`SHORT_LEG = False` holds only the top decile (long-only, directional). Either way the long-leg and
short-leg excess returns are reported separately, so the tradeability of each side is visible.

In [ ]:
def simulate_forward(models, start, end, cost_bps=COST_BPS, short_leg=SHORT_LEG, min_names=MIN_NAMES):
    rebal = pd.date_range(pd.Timestamp(start), pd.Timestamp(end), freq=f'{HORIZON}D')
    recs = []
    for t in rebal:
        probs = score_asof(models, t)
        if len(probs) < min_names: continue
        fwd = pd.Series({s: realized_fwd(s, t) for s in probs.index}).dropna()
        probs = probs.reindex(fwd.index).dropna()
        if len(probs) < min_names: continue
        dec = pd.qcut(probs.rank(method='first'), 10, labels=False)
        long_ret  = fwd[dec == 9].mean()
        short_ret = fwd[dec == 0].mean()
        mkt = fwd.mean()
        if short_leg:
            gross = long_ret - short_ret;         cost = 2 * cost_bps / 1e4
        else:
            gross = long_ret;                     cost = 1 * cost_bps / 1e4     # long-only, one side
        recs.append({'Date': t, 'n': len(fwd), 'long_leg': long_ret - mkt,
                     'short_leg': mkt - short_ret, 'gross': gross, 'net': gross - cost,
                     'longs': list(fwd[dec==9].index), 'shorts': list(fwd[dec==0].index)})
    trades = pd.DataFrame(recs)
    if trades.empty:
        print("no rebalance periods produced -- check dates."); return trades
    r = trades['net']
    ppy = ANNUALIZER / HORIZON
    equity = CAPITAL * (1 + r).cumprod()
    peak = equity.cummax(); dd = (equity/peak - 1).min()
    sharpe = r.mean()/r.std(ddof=1)*np.sqrt(ppy) if r.std(ddof=1) > 0 else np.nan
    total = equity.iloc[-1]/CAPITAL - 1
    n_yrs = (trades['Date'].iloc[-1] - trades['Date'].iloc[0]).days / 365.25
    cagr = (equity.iloc[-1]/CAPITAL)**(1/n_yrs) - 1 if n_yrs > 0 else np.nan
    lsh = trades['long_leg'];  ssh = trades['short_leg']
    def _sh(x): return x.mean()/x.std(ddof=1)*np.sqrt(ppy) if x.std(ddof=1)>0 else np.nan

    print("="*66)
    print(f"FORWARD PAPER SIM  I{N_DAYS}/R{HORIZON}  {trades['Date'].iloc[0].date()} -> "
          f"{trades['Date'].iloc[-1].date()}   ({'LONG-SHORT' if short_leg else 'LONG-ONLY'}, "
          f"{cost_bps}bps/side)")
    print("="*66)
    print(f"  rebalances (periods)     : {len(trades)}")
    print(f"  net Sharpe (ann 365)     : {sharpe:+.2f}")
    print(f"  total return             : {total:+.1%}   (${CAPITAL:,.0f} -> ${equity.iloc[-1]:,.0f})")
    print(f"  CAGR                     : {cagr:+.1%}")
    print(f"  max drawdown             : {dd:+.1%}")
    print(f"  hit rate (periods > 0)   : {(r>0).mean():.0%}")
    print(f"  avg names / rebalance    : {trades['n'].mean():.0f}")
    print(f"  long-leg Sharpe (D9-mkt) : {_sh(lsh):+.2f}")
    print(f"  short-leg Sharpe (mkt-D0): {_sh(ssh):+.2f}   <- shorting small alts is the hard part")
    trades.drop(columns=['longs','shorts']).to_csv('./crypto_sim_trades.csv', index=False)
    print(f"\n  per-period detail -> crypto_sim_trades.csv")

    if HAVE_PLT:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(trades['Date'], equity/CAPITAL, lw=1.6); ax[0].axhline(1, color='grey', lw=.6)
        ax[0].set_title(f"Equity (net, {cost_bps}bps)  Sharpe {sharpe:+.2f}"); ax[0].set_ylabel("x capital")
        ax[1].bar(trades['Date'], trades['net'], width=12); ax[1].axhline(0, color='grey', lw=.6)
        ax[1].set_title("Per-period net return")
        for a in ax: a.grid(alpha=.2)
        plt.tight_layout(); plt.show()
    return trades

trades = simulate_forward(models, SIM_START, SIM_END)

## 5. Today's book — what to actually paper trade now

Prints the coins to long and short at the most recent date in the parquet, and appends them to
`crypto_papertrade_log.csv`. These are entered *now* and held `HORIZON` days. Re-running this over
time (after refreshing the parquet via `crypto_data_prep.ipynb`) accumulates a genuine forward record.

In [ ]:
def todays_book(models, asof=None, decile_frac=0.1, write=True):
    asof = pd.Timestamp(asof or df_market['Date'].max())
    probs = score_asof(models, asof)
    if len(probs) < MIN_NAMES:
        print(f"{asof.date()}: only {len(probs)} names -- too thin to trade."); return None
    k = max(1, int(decile_frac * len(probs)))
    longs  = probs.head(k); shorts = probs.tail(k)
    print(f"BOOK for {asof.date()}  (enter now, hold {HORIZON} days; {len(probs)} coins ranked)\n")
    print(f"  LONG  (top {k}, highest CNN prob-up):")
    for s, p in longs.items():  print(f"     {s:12s} prob_up {p:.3f}")
    print(f"\n  {'SHORT (bottom '+str(k)+', lowest prob-up)' if SHORT_LEG else '(short leg disabled)'}:")
    if SHORT_LEG:
        for s, p in shorts.items(): print(f"     {s:12s} prob_up {p:.3f}")
        print("\n  NOTE: verify each short is actually borrowable/shortable before trusting the short leg.")
    if write:
        rows = [{'asof': asof.date(), 'horizon_days': HORIZON, 'side': 'LONG',
                 'symbol': s, 'prob_up': round(float(p),4), 'weight': round(1.0/k,4)} for s,p in longs.items()]
        if SHORT_LEG:
            rows += [{'asof': asof.date(), 'horizon_days': HORIZON, 'side': 'SHORT',
                      'symbol': s, 'prob_up': round(float(p),4), 'weight': round(-1.0/k,4)} for s,p in shorts.items()]
        newlog = pd.DataFrame(rows)
        if os.path.exists(LOG_CSV):
            old = pd.read_csv(LOG_CSV)
            newlog = pd.concat([old[old['asof'] != str(asof.date())], newlog], ignore_index=True)
        newlog.to_csv(LOG_CSV, index=False)
        print(f"\n  appended {len(rows)} positions -> {LOG_CSV}")
    return probs

_ = todays_book(models)